# Práctica: Conexión a SQLite con Pandas
**Archivo fuente:** `members.csv`  
**Base de datos:** `database_iniciales.db`

## 1. Importación de librerías

In [2]:
import pandas as pd
import sqlite3

## 2. Carga del archivo CSV con Pandas

In [3]:
df = pd.read_csv('members.csv')
print(f'Filas cargadas: {len(df)}')
print(f'Columnas: {list(df.columns)}')
df.head(5)

Filas cargadas: 5000
Columnas: ['participant_id', 'first_name', 'last_name', 'birth_date', 'address', 'phone_number', 'country', 'institute', 'occupation', 'register_time']


,participant_id,first_name,last_name,birth_date,address,phone_number,country,institute,occupation,register_time
0,bd9b6f88-b84f-4c4d-90f8-b67fe2f1a29a,Citra,Nurdiyanti,05-feb-91,"Gg. Monginsidi No. 08\nMedan, Aceh 80734",(0151) 081 2706,Georgia,UD Prakasa Mandasari,Business Intelligence Engineer,1617634046
1,7dfe3391-6f40-47b6-b4db-0c76ebaf5fc3,Aris,Setiawan,11 Jan 1993,"Gg. Rajawali Timur No. 7\nPrabumulih, MA 09434",+62 (036) 461 7027,Korea Utara,Universitas Diponegoro,Frontend Engineer,1617634018
2,19582d7f-b824-4fe5-a517-d5bf573fc768,Cornelia,Handayani,31-jul-93,"Jalan Kebonjati No. 0\nAmbon, SS 57739",089 833 6695,Komoro,UD Hardiansyah Puspasari,Business Analyst,1617634035
3,aeb6d817-25f3-4867-8a74-8d92e0a0f633,Soleh,Rajasa,04-nov-91,"Jl. Yos Sudarso No. 109\nLubuklinggau, SR 76156",+62 (418) 329-4756,Eritrea,Perum Tampubolon Yuliarti,DevOps Engineer,1617634034
4,1fdabdd9-5444-4c97-87b2-fe8833ad0d27,Vivi,Astuti,22 Jan 2003,"Jalan Gardujati No. 53\nKediri, Sulawesi Tenga...",812511835,Aljazair,PT Hardiansyah Rahimah,Data Analyst,1617634010


## 3. Creación de la conexión a SQLite

In [4]:
# Se crea (o abre) la base de datos database_iniciales.db
conn = sqlite3.connect('database_iniciales.db')
print('Conexión establecida con database_iniciales.db')

Conexión establecida con database_iniciales.db


## 4. Escritura del DataFrame en SQLite

In [5]:
# if_exists='replace' recrea la tabla si ya existe
df.to_sql('members', conn, if_exists='replace', index=False)
print('Tabla "members" creada exitosamente.')

Tabla "members" creada exitosamente.


## 5. Visualización del esquema SQL (Schema)

In [6]:
cursor = conn.cursor()
cursor.execute("SELECT sql FROM sqlite_master WHERE type='table' AND name='members';")
schema = cursor.fetchone()[0]
print(schema)

CREATE TABLE "members" (
"participant_id" TEXT,
  "first_name" TEXT,
  "last_name" TEXT,
  "birth_date" TEXT,
  "address" TEXT,
  "phone_number" TEXT,
  "country" TEXT,
  "institute" TEXT,
  "occupation" TEXT,
  "register_time" INTEGER
)


## 6. Consulta de verificación

In [7]:
resultado = pd.read_sql_query('SELECT * FROM members LIMIT 5;', conn)
resultado

,participant_id,first_name,last_name,birth_date,address,phone_number,country,institute,occupation,register_time
0,bd9b6f88-b84f-4c4d-90f8-b67fe2f1a29a,Citra,Nurdiyanti,05-feb-91,"Gg. Monginsidi No. 08\nMedan, Aceh 80734",(0151) 081 2706,Georgia,UD Prakasa Mandasari,Business Intelligence Engineer,1617634046
1,7dfe3391-6f40-47b6-b4db-0c76ebaf5fc3,Aris,Setiawan,11 Jan 1993,"Gg. Rajawali Timur No. 7\nPrabumulih, MA 09434",+62 (036) 461 7027,Korea Utara,Universitas Diponegoro,Frontend Engineer,1617634018
2,19582d7f-b824-4fe5-a517-d5bf573fc768,Cornelia,Handayani,31-jul-93,"Jalan Kebonjati No. 0\nAmbon, SS 57739",089 833 6695,Komoro,UD Hardiansyah Puspasari,Business Analyst,1617634035
3,aeb6d817-25f3-4867-8a74-8d92e0a0f633,Soleh,Rajasa,04-nov-91,"Jl. Yos Sudarso No. 109\nLubuklinggau, SR 76156",+62 (418) 329-4756,Eritrea,Perum Tampubolon Yuliarti,DevOps Engineer,1617634034
4,1fdabdd9-5444-4c97-87b2-fe8833ad0d27,Vivi,Astuti,22 Jan 2003,"Jalan Gardujati No. 53\nKediri, Sulawesi Tenga...",812511835,Aljazair,PT Hardiansyah Rahimah,Data Analyst,1617634010


## 7. Tipos de dato inferidos por Pandas

In [8]:
print(df.dtypes)

participant_id      str
first_name          str
last_name           str
birth_date          str
address             str
phone_number        str
country             str
institute           str
occupation          str
register_time     int64
dtype: object


### Análisis: ¿Por qué `birth_date` es tipo TEXT?

Al inspeccionar los valores del campo `birth_date` se observan formatos inconsistentes:

| Ejemplo       | Formato         |
|---------------|-----------------|
| `05-feb-91`   | DD-MON-YY       |
| `11 Jan 1993` | DD Mon YYYY     |
| `31-jul-93`   | DD-MON-YY       |

Pandas no puede unificar automáticamente formatos de fecha heterogéneos, por lo que infiere el tipo `object` (cadena de texto). SQLite, al recibir ese tipo, lo almacena como `TEXT`. Para convertirlo a un tipo fecha real se requiere `pd.to_datetime()` con el parámetro `format='mixed'`.

### Análisis: Ventaja principal de Pandas en este proceso

Pandas actúa como capa de abstracción entre el archivo plano y la base de datos. Con una sola llamada a `to_sql()` se encarga de:
- Inferir los tipos de dato de cada columna.
- Generar automáticamente el `CREATE TABLE` correspondiente.
- Insertar los registros en lotes eficientes.

Esto elimina la necesidad de escribir manualmente sentencias DDL y múltiples `INSERT INTO`, reduciendo errores y tiempo de desarrollo.

## 8. Cierre de la conexión

In [9]:
conn.close()
print('Conexión cerrada.')

Conexión cerrada.
